<a href="https://colab.research.google.com/github/Jorge-Ruiz-Troccoli/Data-Science-III/blob/main/Practicando_LLM/Practicando_LLM_Groq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚗💨 Predicción de Precios de Autos: de Datos Sucios a Deep Learning

¡Bienvenidos/as! En este notebook vamos a recorrer un pipeline completo de Data Science, de punta a punta:

1. 🧪 **Simulación de datos** (con errores reales de tipeo en las marcas)
2. 🤖 **Normalización inteligente** de texto usando la **API de Groq** (LLM)
3. 🧹 **Data Wrangling** y limpieza
4. 📊 **EDA** (Análisis Exploratorio de Datos)
5. 📈 **Estadística**: correlaciones
6. 📐 **Regresión Lineal Múltiple**
7. 🌲 **Random Forest**
8. 🧠 **Red Neuronal (Deep Learning)**
9. 🏆 **Comparación final de modelos**

**Objetivo:** predecir el `precio_usd` de un auto usado a partir de su marca, modelo, año, kilometraje y color. 🎯

> 💡 **Detalle clave de eficiencia:** en vez de llamarle a la API de Groq **1000 veces** (una por auto), vamos a contar cuántas marcas *únicas* (con errores de tipeo y todo) aparecen realmente en el dataset, y le consultamos a la API **solo esas** — muchas menos llamadas, mismo resultado. 🙌


## 📦 1. Setup


In [ ]:
!pip install -q groq pandas scikit-learn tensorflow matplotlib seaborn


In [ ]:
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams['figure.figsize'] = (10, 6)

print("✅ Setup listo.")

In [ ]:
df= pd.read_csv("https://raw.githubusercontent.com/Jorge-Ruiz-Troccoli/Data-Science-III/refs/heads/main/Practicando_LLM/autos_simulados.csv")
df

## 🤖 2. Normalización de marcas con la API de Groq

Ahora viene la parte de LLM. La idea, paso a paso:

1. Contamos las marcas **únicas** que aparecen en `marca_original` (con `value_counts()`), en vez de mirar las 1000 filas.
2. Le pasamos cada marca única a un LLM (Groq) para que nos devuelva el nombre correcto.
3. Armamos una tabla de traducción (marca sucia → marca limpia) y la aplicamos a **todo** el dataset con `.map()`.

Como en el paso anterior limitamos las variantes sucias a 20 en total, este paso hace como máximo 20 llamadas a la API — sin importar que el dataset tenga 1000 filas.

In [ ]:
conteo_marcas = df['marca_original'].value_counts()
print(f"🔎 Marcas únicas encontradas en el dataset: {len(conteo_marcas)} (sobre {len(df)} filas totales)\n")
print(conteo_marcas)

In [ ]:
from google.colab import userdata
from groq import Groq

client = Groq(api_key=userdata.get('groq_api'))
print("🔑 Cliente de Groq inicializado correctamente.")


In [ ]:
def normalizar_marca_con_groq(nombre_sucio: str) -> str:
    '''Le pide a un LLM (vía Groq) que devuelva el nombre correcto de una marca de auto.'''
    prompt = f'''
    Actúa como un normalizador de datos experto en la industria automotriz.
    Tu tarea es corregir y estandarizar el siguiente nombre de marca de auto.
    Debes devolver ÚNICAMENTE el nombre de la marca correctamente escrito
    (ej. "Toyota", "Chevrolet", "BMW", "Ford", "Volkswagen", "Fiat", "Renault", "Peugeot").
    No agregues texto extra, ni saludos, ni explicaciones, ni puntos finales.

    Nombre a corregir: {nombre_sucio}
    '''
    try:
        completion = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0,
        )
        return completion.choices[0].message.content.strip()
    except Exception as e:
        print(f"⚠️ Error al procesar '{nombre_sucio}': {e}")
        return nombre_sucio  # si falla, dejamos el original

### 🧠 Pausa para entender el `.map()` (y por qué la función va sin paréntesis)

```python
marcas_unicas = conteo_marcas.index.to_series()
marcas_normalizadas = marcas_unicas.map(normalizar_marca_con_groq)
```

**¿Qué hace `.map()`?** `Serie.map(func)` recorre **cada valor** de la Serie y devuelve una Serie nueva (mismo tamaño, mismo índice) donde cada valor es el resultado de aplicarle `func` a ese elemento. Es el equivalente, más prolijo, de escribir:

```python
pd.Series([normalizar_marca_con_groq(v) for v in marcas_unicas], index=marcas_unicas.index)
```

**¿Por qué `normalizar_marca_con_groq` va *sin* paréntesis?** Porque no queremos ejecutarla nosotros ahora mismo — queremos **pasarle la función a `.map()` para que ella la ejecute, una vez por cada elemento**:

- `normalizar_marca_con_groq` (sin `()`) es una *referencia* a la función: un objeto que se puede guardar en una variable o pasar como argumento, igual que pasarías un número o un string. Todavía no se ejecutó nada.
- `normalizar_marca_con_groq()` (con `()`) **ejecuta la función en el momento**, una sola vez. Como la función necesita un argumento (`nombre_sucio`) y no le estaríamos pasando ninguno, esto directamente tiraría un `TypeError`.
- Al escribir `.map(normalizar_marca_con_groq)`, `.map()` internamente hace algo como `normalizar_marca_con_groq(elemento)` por cada `elemento` de la Serie — ahí sí se ejecuta, una vez por cada marca única, con esa marca como argumento.

En resumen: **pasamos la función como un valor** (la "receta"), y es `.map()` quien decide cuándo y con qué dato la va a usar.

In [ ]:
marcas_unicas = conteo_marcas.index.to_series()

print(f"🤖 Consultando la API de Groq para {len(marcas_unicas)} marcas únicas...")
marcas_normalizadas = marcas_unicas.map(normalizar_marca_con_groq)

print("\n✅ ¡Normalización terminada! Mapeo resultante:")
print(marcas_normalizadas)

### 🔁 Aplicando el mapeo a todo el dataset

`marcas_normalizadas` es una Serie indexada por la marca *sucia* (por ejemplo, el índice tiene `"Toyta"` y el valor es `"Toyota"`). Eso es, en la práctica, una tabla de traducción. Cuando hacemos:

```python
df['marca_original'].map(marcas_normalizadas)
```

pandas busca **cada valor** de `marca_original` dentro del *índice* de `marcas_normalizadas` y, si lo encuentra, lo reemplaza por su valor correspondiente (la marca ya corregida). Si alguna marca de `marca_original` no está en ese índice (por ejemplo, si la API falló y no la pudimos normalizar), el resultado para esa fila queda en `NaN`. Por eso encadenamos `.fillna(df['marca_original'])`: para esas filas puntuales, en vez de dejar un hueco, dejamos la marca original sin corregir.

In [ ]:
df['marca_limpia'] = df['marca_original'].map(marcas_normalizadas).fillna(df['marca_original'])

print(f"📊 Marcas únicas antes de normalizar: {df['marca_original'].nunique()}")
print(f"📊 Marcas únicas después de normalizar: {df['marca_limpia'].nunique()}")
df[['marca_original', 'marca_limpia']].head(10)

## 🧹 3. Data Wrangling


In [ ]:
print("📐 Shape:", df.shape)
print("\n🔍 Tipos de datos:")
print(df.dtypes)

In [ ]:
print("❓ Nulos por columna:")
print(df.isna().sum())

print("\n🧬 Duplicados (id_auto):", df['id_auto'].duplicated().sum())

In [ ]:
# Tipos correctos para cada columna
df['marca_limpia'] = df['marca_limpia'].astype('category')
df['modelo'] = df['modelo'].astype('category')
df['color'] = df['color'].astype('category')
df['año'] = df['año'].astype(int)
df['kilometraje'] = df['kilometraje'].astype(int)

print("✅ Tipos de datos actualizados:")
df.dtypes

In [ ]:
print("📊 Estadísticos descriptivos (variables numéricas):")
df.describe()

## 📊 4. EDA — Análisis Exploratorio de Datos


### 💰 Distribución de las variables numéricas principales

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df['precio_usd'], kde=True, ax=axes[0], color='mediumorchid')
axes[0].set_title('💰 Distribución del precio de venta (USD)')

sns.histplot(df['kilometraje'], kde=True, ax=axes[1], color='teal')
axes[1].set_title('🛣️ Distribución del kilometraje')

plt.tight_layout()
plt.show()

### 🚙 ¿Cuántos autos tenemos de cada marca y de qué año?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

orden_marcas = df['marca_limpia'].value_counts().index
sns.countplot(data=df, x='marca_limpia', order=orden_marcas, ax=axes[0], palette='viridis')
axes[0].set_title('🚙 Cantidad de autos por marca')
axes[0].tick_params(axis='x', rotation=30)

sns.countplot(data=df, x='año', ax=axes[1], palette='crest')
axes[1].set_title('📅 Cantidad de autos por año')

plt.tight_layout()
plt.show()

### 💵 Precio según marca y según color

In [ ]:
plt.figure(figsize=(12, 6))
orden_precio = df.groupby('marca_limpia')['precio_usd'].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='marca_limpia', y='precio_usd', order=orden_precio, palette='viridis')
plt.title('🚙 Precio por marca (ya normalizada)')
plt.xlabel('Marca')
plt.ylabel('Precio (USD)')
plt.xticks(rotation=30)
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x='color', y='precio_usd', palette='pastel')
plt.title('🎨 Precio por color de auto')
plt.show()

### 🧩 ¿Qué modelos son los más caros dentro de cada marca?

In [ ]:
precio_por_marca_modelo = df.pivot_table(
    values='precio_usd', index='marca_limpia', columns='modelo', aggfunc='mean'
)

plt.figure(figsize=(12, 6))
sns.heatmap(precio_por_marca_modelo, annot=True, fmt='.0f', cmap='YlGnBu', cbar_kws={'label': 'Precio promedio (USD)'})
plt.title('🧩 Precio promedio por marca y modelo')
plt.xlabel('Modelo')
plt.ylabel('Marca')
plt.show()

### 🛣️ Relación entre kilometraje/año y precio

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.scatterplot(data=df, x='kilometraje', y='precio_usd', hue='marca_limpia', alpha=0.6, ax=axes[0])
axes[0].set_title('🛣️ Kilometraje vs Precio')
axes[0].legend(fontsize=7, loc='upper right')

sns.scatterplot(data=df, x='año', y='precio_usd', hue='marca_limpia', alpha=0.6, ax=axes[1])
axes[1].set_title('📅 Año vs Precio')
axes[1].legend(fontsize=7, loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
precio_promedio_por_año = df.groupby('año')['precio_usd'].mean().reset_index()

plt.figure(figsize=(10, 5))
sns.lineplot(data=precio_promedio_por_año, x='año', y='precio_usd', marker='o', color='darkorange')
plt.title('📈 Evolución del precio promedio según año del auto')
plt.ylabel('Precio promedio (USD)')
plt.show()

### 🔬 Vista multivariada (pairplot)

In [ ]:
sns.pairplot(
    df[['año', 'kilometraje', 'precio_usd', 'marca_limpia']],
    hue='marca_limpia', diag_kind='kde', plot_kws={'alpha': 0.5}, height=2.3,
)
plt.suptitle('🔬 Relación entre año, kilometraje y precio, por marca', y=1.02)
plt.show()

## 📈 5. Estadística: Matriz de correlación

Miramos qué tan (linealmente) relacionadas están las variables numéricas con el precio. El **coeficiente de correlación de Pearson** va de -1 a 1: cerca de 1 es relación positiva fuerte, cerca de -1 negativa fuerte, y cerca de 0 no hay relación lineal.

In [ ]:
num_cols = ['año', 'kilometraje', 'precio_usd']
corr = df[num_cols].corr()

plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt='.2f')
plt.title('🔥 Matriz de correlación de Pearson')
plt.show()

In [ ]:
print("📌 Coeficientes de correlación con el precio:")
for col in ['año', 'kilometraje']:
    r = corr.loc[col, 'precio_usd']
    signo = "positiva 📈" if r > 0 else "negativa 📉"
    print(f"   • {col} vs precio_usd: r = {r:.3f} (correlación {signo})")

## 🛠️ 6. Preparación de datos para Machine Learning


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# One-hot encoding de las variables categóricas
X = pd.get_dummies(
    df[['año', 'kilometraje', 'marca_limpia', 'color']],
    columns=['marca_limpia', 'color'],
    drop_first=True,
)
y = df['precio_usd']

print(f"🧩 Features usadas ({X.shape[1]}): {list(X.columns)}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=20)

print(f"🏋️ Train: {X_train.shape[0]} autos | 🧪 Test: {X_test.shape[0]} autos")

In [ ]:
# Escalado de features (necesario sobre todo para la red neuronal)
scaler_X = StandardScaler()
X_train_s = scaler_X.fit_transform(X_train)
X_test_s = scaler_X.transform(X_test)

In [ ]:
# Escalado del target (ayuda mucho a que la red neuronal converja más rápido)
scaler_y = StandardScaler()
y_train_s = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).flatten()
y_test_s = scaler_y.transform(y_test.values.reshape(-1, 1)).flatten()

## 📐 7. Regresión Lineal Múltiple

Empezamos con el modelo más simple e interpretable: una **regresión lineal múltiple**. Antes de entrenarla, armamos una **matriz de correlación con heatmap** de todas las features (incluyendo las variables dummy de marca y color) contra el precio, para entender qué variables pesan más. Después entrenamos el modelo con `scikit-learn` y medimos sus métricas en test.


In [ ]:
# Matriz de correlación de TODAS las features (incluye dummies de marca/color) vs precio
corr_features = X_train.astype(float).copy()
corr_features['precio_usd'] = y_train.values
corr_matrix_full = corr_features.corr()

plt.figure(figsize=(12, 9))
sns.heatmap(corr_matrix_full, cmap='coolwarm', vmin=-1, vmax=1, cbar_kws={'label': 'Correlación de Pearson'})
plt.title('🔥📐 Matriz de correlación — Features vs Precio (Regresión Lineal)')
plt.tight_layout()
plt.show()

print("📌 Top 10 features más correlacionadas con el precio (valor absoluto):")
top_corr = corr_matrix_full['precio_usd'].drop('precio_usd').abs().sort_values(ascending=False).head(10)
print(top_corr)


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

lr = LinearRegression()
lr.fit(X_train, y_train)
pred_lr = lr.predict(X_test)

In [ ]:
rmse_lr = mean_squared_error(y_test, pred_lr) ** 0.5
mae_lr = mean_absolute_error(y_test, pred_lr)
r2_lr = r2_score(y_test, pred_lr)

print("📐 Regresión Lineal Múltiple — métricas en test:")
print(f"   RMSE: {rmse_lr:,.2f} USD")
print(f"   MAE:  {mae_lr:,.2f} USD")
print(f"   R²:   {r2_lr:.4f}")

## 🌲 8. Random Forest Regressor


In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=300, max_depth=None, random_state=20, n_jobs=-1)
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)

In [ ]:
rmse_rf = mean_squared_error(y_test, pred_rf) ** 0.5
mae_rf = mean_absolute_error(y_test, pred_rf)
r2_rf = r2_score(y_test, pred_rf)

print("🌲 Random Forest — métricas en test:")
print(f"   RMSE: {rmse_rf:,.2f} USD")
print(f"   MAE:  {mae_rf:,.2f} USD")
print(f"   R²:   {r2_rf:.4f}")

In [ ]:
importancias = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(9, 6))
sns.barplot(x=importancias.values[:10], y=importancias.index[:10], palette='crest')
plt.title('🌟 Importancia de features — Random Forest')
plt.xlabel('Importancia')
plt.show()

## 🧠 9. Red Neuronal (Deep Learning)

Por último, entrenamos una **red neuronal feed-forward** con Keras/TensorFlow: 2 capas ocultas con activación ReLU. Usamos `EarlyStopping` para frenar el entrenamiento apenas deja de mejorar en validación (evita overfitting y nos ahorra tiempo ⏱️).

In [ ]:
import tensorflow as tf

tf.random.set_seed(20)

nn_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train_s.shape[1],)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1),
])

nn_model.summary()

In [ ]:
nn_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.01), loss='mse', metrics=['mae'])

early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

In [ ]:
history = nn_model.fit(
    X_train_s, y_train_s,
    validation_split=0.2,
    epochs=200,
    batch_size=16,
    callbacks=[early_stop],
    verbose=0,
)

print(f"🏁 Entrenamiento terminado en {len(history.history['loss'])} épocas (gracias, EarlyStopping 🙏)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'], label='Loss (train) 🏋️')
axes[0].plot(history.history['val_loss'], label='Loss (validation) 🧪')
axes[0].set_title('📉 Curva de pérdida (MSE)')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('MSE (escalado)')
axes[0].legend()

axes[1].plot(history.history['mae'], label='MAE (train) 🏋️')
axes[1].plot(history.history['val_mae'], label='MAE (validation) 🧪')
axes[1].set_title('📉 Curva de MAE')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('MAE (escalado)')
axes[1].legend()

plt.suptitle('🧠 Curvas de aprendizaje — Detección de Underfitting / Overfitting', y=1.02)
plt.tight_layout()
plt.show()

# 🔍 Diagnóstico automático de ajuste del modelo
loss_train_final = history.history['loss'][-1]
loss_val_final = history.history['val_loss'][-1]
gap_pct = (loss_val_final - loss_train_final) / loss_train_final * 100

print("🔍 Diagnóstico de ajuste del modelo:")
print(f"   • Loss train final: {loss_train_final:.4f} | Loss validation final: {loss_val_final:.4f}")
if gap_pct > 20:
    print(f"   ⚠️ Posible OVERFITTING: la pérdida de validación es {gap_pct:.1f}% mayor que la de entrenamiento.")
elif loss_train_final > 0.3:
    print(f"   ⚠️ Posible UNDERFITTING: la pérdida de entrenamiento sigue alta ({loss_train_final:.3f}), el modelo no terminó de aprender el patrón.")
else:
    print(f"   ✅ Buen ajuste: train y validación evolucionan parejo (gap: {gap_pct:.1f}%).")


In [ ]:
pred_nn_s = nn_model.predict(X_test_s, verbose=0).flatten()
pred_nn = scaler_y.inverse_transform(pred_nn_s.reshape(-1, 1)).flatten()

In [ ]:
rmse_nn = mean_squared_error(y_test, pred_nn) ** 0.5
mae_nn = mean_absolute_error(y_test, pred_nn)
r2_nn = r2_score(y_test, pred_nn)

print("🧠 Red Neuronal — métricas en test:")
print(f"   RMSE: {rmse_nn:,.2f} USD")
print(f"   MAE:  {mae_nn:,.2f} USD")
print(f"   R²:   {r2_nn:.4f}")

## 🏆 10. Comparación final de modelos


In [ ]:
resultados = pd.DataFrame({
    'Modelo': ['Regresión Lineal 📐', 'Random Forest 🌲', 'Red Neuronal 🧠'],
    'RMSE': [rmse_lr, rmse_rf, rmse_nn],
    'MAE': [mae_lr, mae_rf, mae_nn],
    'R2': [r2_lr, r2_rf, r2_nn],
}).sort_values('RMSE')

print("📋 Tabla comparativa de modelos:")
resultados

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=resultados, x='Modelo', y='RMSE', palette='rocket', ax=axes[0])
axes[0].set_title('💥 RMSE por modelo (menor es mejor)')

sns.barplot(data=resultados, x='Modelo', y='R2', palette='mako', ax=axes[1])
axes[1].set_title('🎯 R² por modelo (más cerca de 1 es mejor)')

plt.tight_layout()
plt.show()

ganador = resultados.iloc[0]['Modelo']
print(f"🏆 El modelo con menor error (RMSE) fue: {ganador}")

## 🎯 11. Conclusiones

- Normalizar texto "sucio" con un LLM (Groq) es muy eficiente si primero **deduplicamos** las variantes (`value_counts()`) en vez de consultar fila por fila. 🤖✂️
- `.map()` es la herramienta clave para aplicar esa normalización: pasarle la función *sin* paréntesis es lo que le permite a pandas ejecutarla automáticamente sobre cada elemento.
- El **año** y el **kilometraje** tienen una correlación fuerte y esperable con el precio (a más antigüedad/kilometraje, menor precio). 📉
- La **regresión lineal** es un gran baseline interpretable, mientras que **Random Forest** y la **red neuronal** pueden capturar relaciones no lineales entre marca, modelo, año, kilometraje y color.
- Ningún modelo es "el mejor" en todos los escenarios: depende del tamaño de datos, del ruido y de cuánto valoramos la interpretabilidad vs. la performance pura. ⚖️

¡Gracias por seguir el notebook hasta acá! 🙌🚗💨
